In [2]:
!pip -q install -U transformers accelerate

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16 if device=="cuda" else torch.float32).to(device)

model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [4]:
print(f"Model Loaded on: {device}")

Model Loaded on: cuda


In [12]:
messages = [
    {
        "role" : "system",
        "content" : "You are a friendly and consice AI assistant"
    }
]

def conversation(user_message):
  messages.append(
      {
          "role" : "user",
          "content" : user_message
      }
  )

  inputs = tokenizer.apply_chat_template(messages, tokenize = True, add_generation_prompt = True, return_tensors = 'pt', return_dict=True).to(device)

  with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens = 120, do_sample = False, pad_token_id =tokenizer.eos_token_id)

  new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]

  answer = tokenizer.decode(
      new_tokens,
      skip_special_tokens = True
  )

  messages.append(
      {
          "role": "assistant",
          "content" : answer
      }
  )

  return answer

In [13]:
print(conversation("Hello! My name is Maria"))

Hello Maria! How can I assist you today?


In [14]:
print(conversation("I want to know about Machine Learning. Can you please explain briefly?"))

Certainly! Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. These systems can then use this learned patterns to make predictions or decisions based on new, unseen data. Common applications include image recognition, speech recognition, recommendation systems, and predictive analytics. Machine learning techniques are used in various fields such as healthcare, finance, transportation, and more.


In [15]:
print(conversation("Got it. This is helpful, thanks"))

You're welcome! If you have any other questions, feel free to ask.


In [16]:
print(conversation("Sure, thanks"))

You're welcome! Don't hesitate to ask if you need further assistance. Have a great day!
